Chuyển file docx -> md

In [ ]:
!pip install pypandoc

^C



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pypandoc
pypandoc.download_pandoc()

In [4]:
import os
import pypandoc

input_dir = "docs"
output_dir = "md"

os.makedirs(output_dir, exist_ok=True)

for filename in os.listdir(input_dir):
    if filename.endswith(".docx"):
        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(
            output_dir,
            filename.replace(".docx", ".md")
        )

        pypandoc.convert_file(
            input_path,
            'md',
            outputfile=output_path
        )

        print(f"Converted: {filename}")

Converted: chinhSachBaoHanh.docx
Converted: chinhSachBaoMat.docx
Converted: chinhSachDoiTra.docx
Converted: chinhSachVanChuyen.docx


LÀM SẠCH DATA

In [ ]:
import re

def clean_basic(text: str) -> str:
    # bỏ ký tự lạ
    text = text.replace("\xa0", " ")
    # bỏ dòng trang trí ===== -----
    text = re.sub(r"^[=\-]{3,}$", "", text, flags=re.MULTILINE)
    # chuẩn hóa xuống dòng
    text = re.sub(r"\n{3,}", "\n\n", text)
    # chuẩn hóa khoảng trắng
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()

In [6]:
def is_table(block: str) -> bool:
    return "|" in block and "---" in block

In [7]:
def table_to_text(table_md: str) -> str:
    lines = [l for l in table_md.splitlines() if "|" in l]
    rows = []

    for line in lines[2:]:  # bỏ header
        cells = [c.strip() for c in line.split("|") if c.strip()]
        rows.append(" - ".join(cells))

    return "\n".join(rows)

In [8]:
from pathlib import Path

input_dir = Path("data/raw_md")
output_dir = Path("data/clean_md")
output_dir.mkdir(exist_ok=True)

for md_file in input_dir.glob("*.md"):
    text = md_file.read_text(encoding="utf-8")

    # clean cơ bản
    text = clean_basic(text)

    # xử lý theo block
    blocks = text.split("\n\n")
    new_blocks = []

    for block in blocks:
        if is_table(block):
            new_blocks.append(table_to_text(block))
        else:
            new_blocks.append(block)

    cleaned_text = "\n\n".join(new_blocks)

    output_path = output_dir / md_file.name
    output_path.write_text(cleaned_text, encoding="utf-8")

    print(f"Cleaned: {md_file.name}")

Cleaned: chinhSachBaoHanh.md
Cleaned: chinhSachBaoMat.md
Cleaned: chinhSachDoiTra.md
Cleaned: chinhSachVanChuyen.md


CHIA VĂN BẢN THÀNH CÁC ĐOẠN NHỎ (CHUNKING)

In [7]:
#CHUNK THEO Ý NGHĨA
import re

def data_chunks_Warranty(text):
  with open(text, "r", encoding="utf-8") as f:
    text = f.read()
  docs=[]
  pattern = r"\n(?=\d+\.\s+[A-ZÀ-Ỹ])"
  section = re.split(pattern, text.strip())
  for i, s in enumerate(section, 1):
    title = s.split("\n")[0].strip()
    content = s.strip()
    docs.append({
       "id": f"warranty_{i}",
            "content": content,
            "metadata": {
                "policy": "warranty",
                "section": title.lower().replace(" ", "_")
          }
    })
  return docs

In [8]:
data_chunks_Warranty("D:\HOC\CDIO3\XULYDATA_CRM\data\clean_md\chinhSachBaoHanh.md")

[{'id': 'warranty_1',
  'content': 'CHÍNH SÁCH BẢO HÀNH GEARVN\n\n*(Cập nhật ngày 22/03/2024)*\n\n1\\. KÊNH TIẾP NHẬN VÀ LIÊN HỆ BẢO HÀNH\n\nKhi có nhu cầu bảo hành, quý khách vui lòng liên hệ qua các hình thức\nsau:\n\n- Tổng đài bảo hành:\n [1900.5325](https://www.google.com/search?q=tel:19005325&authuser=2)\n\n- Trực tuyến: Nhắn tin trực tiếp tại Website hoặc Fanpage của GEARVN.\n\n- Trực tiếp: Mang sản phẩm đến các chi nhánh cửa hàng/Trung tâm bảo hành\n của GEARVN hoặc Trung tâm bảo hành của hãng.\n\nLưu ý khi gửi hàng qua đơn vị vận chuyển:\n\n- Khách hàng có thể gửi sản phẩm bảo hành qua bưu điện/chuyển phát\n nhanh. GEARVN sẽ phản hồi sau khi tiếp nhận.\n\n- Vui lòng liên hệ nhân viên tư vấn để được hướng dẫn trước khi gửi\n hàng.\n\n2\\. ĐIỀU KIỆN ĐƯỢC BẢO HÀNH\n\nSản phẩm sẽ được chấp nhận bảo hành nếu đáp ứng các tiêu chí sau:',
  'metadata': {'policy': 'warranty', 'section': 'chính_sách_bảo_hành_gearvn'}},
 {'id': 'warranty_2',
  'content': '1. Thời hạn: Sản phẩm còn trong 

In [13]:
def data_chunks_Privacy(text):
  with open(text, "r", encoding="utf-8") as f:
    text = f.read()
  docs=[]
  pattern = r"\n(?=\d+\.\s+[A-ZÀ-Ỹ])"
  section = re.split(pattern, text.strip())
  for i, s in enumerate(section, 1):
    title = s.split("\n")[0].strip()
    content = s.strip()
    docs.append({
       "id": f"privacy_{i}",
            "content": content,
            "metadata": {
                "policy": "privacy",
                "section": title.lower().replace(" ", "_")
          }
    })
  return docs

In [14]:
data_chunks_Privacy("D:\HOC\CDIO3\XULYDATA_CRM\data\clean_md\chinhSachBaoMat.md")

[{'id': 'privacy_1',
  'content': '**CHÍNH SÁCH BẢO MẬT THÔNG TIN KHÁCH HÀNG GEARVN**\n\n1\\. MỤC ĐÍCH VÀ PHẠM VI THU THẬP THÔNG TIN\n\nGEARVN cam kết không bán, chia sẻ hay trao đổi thông tin cá nhân của\nkhách hàng cho bất kỳ bên thứ ba nào khác vì mục đích thương mại. Thông\ntin cá nhân thu thập được sẽ chỉ được sử dụng trong nội bộ công ty.\n\nKhi quý khách liên hệ đăng ký dịch vụ, các thông tin mà GEARVN thu thập\nbao gồm:\n\n- Thông tin cá nhân: Họ và tên, Địa chỉ, Số điện thoại, Email.\n\n- Thông tin dịch vụ: Tên sản phẩm, Số lượng, Thời gian giao nhận sản\n phẩm.\n\n2\\. PHẠM VI SỬ DỤNG THÔNG TIN\n\nThông tin cá nhân thu thập được sẽ chỉ được GEARVN sử dụng trong nội bộ\ncông ty cho các mục đích sau:\n\n- Hỗ trợ khách hàng.\n\n- Cung cấp thông tin liên quan đến dịch vụ.\n\n- Xử lý đơn đặt hàng và cung cấp dịch vụ theo yêu cầu của bạn.\n\n- Gửi thông tin sản phẩm, dịch vụ mới, sự kiện hoặc tuyển dụng (nếu quý\n khách có đăng ký nhận email thông báo).\n\nTrường hợp ngoại lệ (Tuân

In [20]:
def data_chunks_Return(text):
  with open(text, "r", encoding="utf-8") as f:
    text = f.read()
  docs=[]
  pattern = r"\n(?=\d+\.\s+[A-ZÀ-Ỹ])"
  section = re.split(pattern, text.strip())
  for i, s in enumerate(section, 1):
    title = s.split("\n")[0].strip()
    content = s.strip()
    docs.append({
       "id": f"return_{i}",
            "content": content,
            "metadata": {
                "policy": "return",
                "section": title.lower().replace(" ", "_")
          }
    })
  return docs

In [21]:
data_chunks_Return("D:\HOC\CDIO3\XULYDATA_CRM\data\clean_md\chinhSachDoiTra.md")

[{'id': 'return_1',
  'content': 'CHÍNH SÁCH ĐỔI TRẢ SẢN PHẨM GEARVN\n\n1\\. ĐIỀU KIỆN ĐỔI TRẢ\n\nQuý khách hàng cần kiểm tra tình trạng hàng hóa và có thể yêu cầu đổi\nhàng/trả lại hàng ngay tại thời điểm giao/nhận hàng trong những trường\nhợp sau:\n\n- Sai lệch sản phẩm: Hàng không đúng chủng loại, mẫu mã trong đơn hàng\n đã đặt hoặc như mô tả trên website tại thời điểm đặt hàng.\n\n- Thiếu hụt: Không đủ số lượng, không đủ bộ như trong đơn hàng.\n\n- Hư hỏng ngoại quan: Tình trạng bên ngoài bị ảnh hưởng như rách bao bì,\n bong tróc, bể vỡ...\n\nLưu ý: Khách hàng có trách nhiệm trình giấy tờ liên quan chứng minh sự\nthiếu sót trên để hoàn thành việc hoàn trả/đổi trả hàng hóa.\n\n2\\. QUY ĐỊNH VỀ THỜI GIAN VÀ CÁCH THỨC ĐỔI TRẢ\n\n- Thời gian thông báo đổi trả: Trong vòng 48 giờ kể từ khi nhận sản phẩm\n (đối với trường hợp sản phẩm thiếu phụ kiện, quà tặng hoặc bể vỡ).\n\n- Thời gian gửi chuyển trả sản phẩm: Trong vòng 14 ngày kể từ khi nhận\n sản phẩm.\n\n- Địa điểm đổi trả sản phẩm: 

In [22]:
def data_chunks_Shipping(text):
  with open(text, "r", encoding="utf-8") as f:
    text = f.read()
  docs=[]
  pattern = r"\n(?=\d+\.\s+[A-ZÀ-Ỹ])"
  section = re.split(pattern, text.strip())
  for i, s in enumerate(section, 1):
    title = s.split("\n")[0].strip()
    content = s.strip()
    docs.append({
       "id": f"shipping_{i}",
            "content": content,
            "metadata": {
                "policy": "shipping",
                "section": title.lower().replace(" ", "_")
          }
    })
  return docs

In [23]:
data_chunks_Shipping("D:\HOC\CDIO3\XULYDATA_CRM\data\clean_md\chinhSachVanChuyen.md")

[{'id': 'shipping_1',
  'content': 'CHÍNH SÁCH VẬN CHUYỂN GEARVN\n\n*(Có hiệu lực từ ngày 03/09/2025)*\n\nGEARVN cung cấp dịch vụ giao hàng toàn quốc, gửi hàng tận nơi đến địa\nchỉ của Quý khách. Thời gian giao hàng dự kiến phụ thuộc vào kho hàng và\nđịa chỉ nhận hàng.\n\n1\\. PHÍ DỊCH VỤ GIAO HÀNG (GIAO NHANH 2H - 4H)\n\nÁp dụng cho khu vực Hồ Chí Minh và Hà Nội:\n\n ----------------------------------------------------------------------\n Giá trị đơn hàng Khu vực HCM / Hà Nội Khu vực Ngoại thành / Tỉnh\n ------------------ ----------------------- ---------------------------\n Dưới 5 triệu đồng 40.000 VNĐ Không áp dụng\n\n Trên 5 triệu đồng Miễn phí Không áp dụng\n ----------------------------------------------------------------------\n\n2\\. THỜI GIAN GIAO HÀNG DỰ KIẾN\n\nThời gian giao hàng tiêu chuẩn được tính dựa trên tuyến đường vận\nchuyển:\n\nNội tỉnh/Thành phố - Nội thành & Ngoại - 1 - 2 ngày\nthành\n(HCM -- HCM / HN -- HN)\nLiên vùng lân cận - Trung tâm Tỉnh/TP - 3 - 4 ngày\n\

VECTOR EMBEDDING 